In [ ]:
import os, sys, shutil
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader

os.system(f'git clone https://github.com/m-j-r-q/Cura.git /kaggle/working/Cura')
sys.path.append('/kaggle/working/Cura/backend/src')

IMAGE_DIR  = '/kaggle/input/datasets/khanfashee/nih-chest-x-ray-14-224x224-resized/images-224/images-224'
DATA_DIR   = '/kaggle/input/datasets/junaidrasul/cura-metadata'
OUTPUT_DIR = '/kaggle/working/outputs'
os.makedirs(f'{OUTPUT_DIR}/checkpoints', exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
!git -C /kaggle/working/Cura pull

In [ ]:
from model import build_model
from train_run import train_model
from utils import load_pos_weights

def resume_training(architecture, checkpoint_filename, epochs=3, lr=5e-5, batch_size=32):
    model = build_model(architecture, pretrained=False).to(device)
    
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs.")
        model = nn.DataParallel(model)

    optimizer = torch.optim.Adam(
        model.module.parameters() if isinstance(model, nn.DataParallel) else model.parameters(),
        lr=lr
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=2, factor=0.5
    )

    checkpoint_path = f'{DATA_DIR}/{checkpoint_filename}'
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint['model_state_dict'])

    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    start_epoch = checkpoint['epoch'] + 1
    best_auc    = checkpoint['best_auc']

    print(f"Resumed {architecture} from epoch {start_epoch}")
    print(f"Best AUC so far: {best_auc:.4f}")

    model = train_model(
        architecture=architecture,
        data_dir=DATA_DIR,
        image_dir=IMAGE_DIR,
        output_dir=OUTPUT_DIR,
        epochs=epochs,
        lr=lr,
        pretrained_model=model,
        initial_best_auc=best_auc
    )

    return model

In [ ]:
# DenseNet121
model = resume_training(
    architecture='densenet121',
    checkpoint_filename='densenet121_checkpoint.pt',
    epochs=10,
    lr=5e-5,
    batch_size=32
)

In [ ]:
# EfficientNet-B0
model = resume_training(
    architecture='efficientnet_b0',
    checkpoint_filename='efficientnet_b0_checkpoint.pt',
    epochs=10,
    lr=5e-5,
    batch_size=32,
)

In [ ]:
# ResNet50
model = resume_training(
    architecture='resnet50',
    checkpoint_filename='resnet50_checkpoint.pt',
    epochs=10,
    lr=5e-5,
    batch_size=32
)